# Wang task structure and model-validity simulations

*Draft introduction. The scope and summary will be revised as the simulation sections are completed.*

This notebook extends the simulation framework developed in Notebook 6 to the task structure described by Wang et al. (2026). It asks whether the estimator can recover a genuine within-participant mood × RPE interaction under this task structure, and whether alternative processes could produce misleading interaction estimates.

The planned simulations cover clean-data recovery, incomplete reward histories caused by omitted attention-check trials, non-linear use of the bounded happiness scale, and whether preceding mood changes responsiveness specifically to reward prediction errors or to reward outcomes more generally. A conditional extension considers heterogeneous RPE slopes combined with mood-dependent gambling.

Using synthetic data only, these simulations inform the final analysis specification and the interpretation of subsequent empirical findings.


## 0. Estimation procedure and task specification

### 0.1 Estimation procedure

A shared estimator was adapted from Notebook 6. The model was fitted by ordinary least squares after subtracting participant means from the outcome and predictors, including the interaction terms. Standard errors were clustered by participant, with a finite-sample correction applied to the covariance matrix.

The extracted estimator was checked against the original implementation using fixed-seed synthetic datasets with and without mood-dependent RPE sensitivity. Coefficient estimates, standard errors, and confidence intervals agreed within numerical tolerance, including when participants contributed unequal numbers of observations. Additional tests confirmed agreement with an equivalent regression using explicit participant intercepts.

The implementation is available in [estimator.py](estimator.py), with [verification tests](../tests/test_estimator.py).

### 0.2 Experimental task specification

The task specification was based on the online risky-choice experiment described by Wang et al. (2026, pp. 13–14), supplemented by clarification from the author.

| Feature | Specification | Status |
|---|---|---|
| Regular trials | 90 trials: 30 gain, 30 loss, and 30 mixed, presented in random order. | Documented |
| Gamble probabilities | Two equally probable outcomes per gamble. | Documented |
| Gain and loss trials | Certain amounts of ±35, ±45, or ±55 points versus gambles yielding zero or an amount determined by one of ten published multipliers. | Documented |
| Mixed trials | Certain outcome of zero versus a gamble involving a gain of 40, 45, or 75 points and a loss determined by one of ten published multipliers. | Documented |
| Happiness ratings | An initial rating followed by ratings every two or three trials, on a 0–100 scale. | Documented |
| Slider interface | No preset anchor; the handle appeared only after the first interaction. | Documented; author clarification |
| Attention checks | Four additional trials contributing to the cumulative points total; none immediately preceded a happiness rating. Exact outcomes and positions unavailable. | Documented; author clarification; unresolved details |
| Trial allocation and rating schedule | The exact distribution of reward combinations and positions of happiness ratings are not specified in the published methods. | Not reported |

Interface behaviour and attention-check details were clarified by Z. Wang (personal communication, October 31 and November 20, 2026).

The published reward amounts and multipliers will be used to construct the synthetic task. Where the exact allocation of trials or ratings is not documented, simulation assumptions will be stated explicitly. The published mixed-trial gain amounts will be checked against the released data after ethics approval.

### 0.3 Simulation parameters

*Generating parameter values, units, and their sources or assumptions will be documented here as the simulation design is implemented. The participant's generating mood reference level and initial happiness rating will be specified separately.*

In [6]:
from estimator import fit_intervals

# no additive mood_between term: participant intercepts absorb this constant
# the mood_between × RPE interaction remains among the predictors
predictor_names = [
    "mood_within",
    "CR", "EV", "RPE",
    "within_x_RPE",
    "between_x_RPE",
]


## 1. Synthetic Wang task generator

*Generator in development* How closely can the synthetic task reproduce the documented Wang experiment, and which unresolved task features require explicit assumptions?


### 1.1 Regular trial offers

The simulated offer set included each combination of the three amounts and ten multipliers once within each trial type, giving 30 gain, 30 loss and 30 mixed trials. This allocation was adopted as a simulation assumption, based on the published counts and consistent with the stated purpose of accommodating a wide range of risk and loss sensitivity (Wang et al., 2026, pp. 13–14). Each simulated participant received the same set of offers in an independently randomised order. Calculated amounts were retained without rounding.

| Feature | Specification | Status |
|---|---|---|
| Offer allocation | Each amount–multiplier combination appears once per trial type. | Assumed; checkable after approval. |
| Presentation order | Independent shuffling of the regular offers for each participant. | Random presentation is documented; independent participant-specific shuffling is an implementation choice. |
| Rounding | Calculated amounts are retained without rounding. | Assumed; checkable after approval. |
| Mixed-trial gains | 40, 45 and 75 points. | Documented; possible error to be checked after approval. |

The table below contains the regular offers before randomisation. Attention checks, choices, outcomes and rating boundaries were added in subsequent steps.


In [7]:
import pandas as pd

certain_amounts = [35, 45, 55]
gain_loss_multipliers = [1.68, 1.82, 2, 2.22, 2.48, 2.8, 3.16, 3.6, 4.2, 5]
mixed_gains = [40, 45, 75]
mixed_loss_multipliers = [0.2, 0.34, 0.5, 0.64, 0.77, 0.89, 1, 1.1, 1.35, 2]

offer_rows = []

# each certain amount is paired with every multiplier, for gains and losses
for amount in certain_amounts:
    for multiplier in gain_loss_multipliers:
        offer_rows.append({
            "trial_type": "gain",
            "certain_value": amount,
            "win_value": amount * multiplier,
            "lose_value": 0,
        })
        offer_rows.append({
            "trial_type": "loss",
            "certain_value": -amount,
            "win_value": 0,
            "lose_value": -amount * multiplier,
        })

# mixed trials offer zero for certain, or a positive gain and a negative loss
for gain in mixed_gains:
    for multiplier in mixed_loss_multipliers:
        offer_rows.append({
            "trial_type": "mixed",
            "certain_value": 0,
            "win_value": gain,
            "lose_value": -gain * multiplier,
        })

regular_offers = pd.DataFrame(offer_rows)
regular_offers["win_probability"] = 0.5

# identify offers before shuffling; this is not their presentation order
regular_offers.insert(0, "offer_id", range(1, len(regular_offers) + 1))

regular_offers.head(6)


,offer_id,trial_type,certain_value,win_value,lose_value,win_probability
0,1,gain,35,58.8,0.0,0.5
1,2,loss,-35,0.0,-58.8,0.5
2,3,gain,35,63.7,0.0,0.5
3,4,loss,-35,0.0,-63.7,0.5
4,5,gain,35,70.0,0.0,0.5
5,6,loss,-35,0.0,-70.0,0.5


In [8]:
import numpy as np

# 30 offers per type
assert regular_offers.groupby("trial_type").size().to_dict() == {
    "gain": 30, "loss": 30, "mixed": 30,
}

# no repeated offers within a trial type
assert not regular_offers.duplicated(
    ["trial_type", "certain_value", "win_value", "lose_value"]
).any()

# check one fractional mixed loss: 40 × 0.34 = 13.6, with no rounding
mixed_offers = regular_offers[regular_offers["trial_type"] == "mixed"]
assert np.isclose(mixed_offers.loc[
    mixed_offers["win_value"] == 40, "lose_value"
], -13.6).sum() == 1

regular_offers.groupby("trial_type").size().rename("offers")


trial_type
gain     30
loss     30
mixed    30
Name: offers, dtype: int64

### 1.2 Presentation order and random streams

The regular offers were randomised separately for each simulated participant. A fixed seed and replication index determined four random streams for task construction, choices, mood innovations and measurement. Separating these streams allows a component to change without altering the random draws used by other components. The example below constructs regular-trial sequences for two participants; only the task stream is used at this stage.


In [9]:
# one reproducible set of component streams for this replication
base_seed = 20261122
replication = 0
replication_seed = np.random.SeedSequence([base_seed, replication])
task_seed, choice_seed, mood_seed, measurement_seed = replication_seed.spawn(4)

task_rng = np.random.default_rng(task_seed)
# the other three seeds are reserved for their respective simulation components


In [10]:
participant_tables = []

for participant_id in [1, 2]:
    # draw a new order without resetting the random-number generator
    order = task_rng.permutation(len(regular_offers))
    participant_offers = regular_offers.iloc[order].reset_index(drop=True)

    participant_offers.insert(0, "participant_id", participant_id)
    participant_offers.insert(
        1, "regular_trial_order", range(1, len(participant_offers) + 1)
    )
    participant_tables.append(participant_offers)

# attention checks will be inserted later, changing the complete task order
participant_trials = pd.concat(participant_tables, ignore_index=True)

# confirm that shuffling preserves every original offer and its amounts
for participant_id, trials in participant_trials.groupby("participant_id"):
    restored_offers = (
        trials.sort_values("offer_id")[regular_offers.columns]
        .reset_index(drop=True)
    )
    pd.testing.assert_frame_equal(restored_offers, regular_offers)

# inspect the first five regular offers for each participant
participant_trials.groupby("participant_id").head(5)


,participant_id,regular_trial_order,offer_id,trial_type,certain_value,win_value,lose_value,win_probability
0,1,1,6,loss,-35,0.0,-70.00,0.5
1,1,2,41,gain,55,92.4,0.00,0.5
2,1,3,37,gain,45,189.0,0.00,0.5
3,1,4,3,gain,35,63.7,0.00,0.5
4,1,5,38,loss,-45,0.0,-189.00,0.5
90,2,1,53,gain,55,173.8,0.00,0.5
91,2,2,25,gain,45,90.0,0.00,0.5
92,2,3,35,gain,45,162.0,0.00,0.5
93,2,4,76,mixed,0,45.0,-40.05,0.5
94,2,5,54,loss,-55,0.0,-173.80,0.5


### 1.3 Rating schedule and attention-check positions

The default schedule comprised 36 intervals containing two or three regular trials, with four attention checks inserted within these intervals. Both the interval count and the counting rule are parameters: 35 intervals and a rule counting all 94 trials were also implemented. An initial rating was assumed to precede all events, and the final rating was assumed to follow the last trial. These schedule choices remain provisional pending access to the data.

The author confirmed that attention checks contribute to cumulative points and never immediately precede a rating. Under regular-trial counting, each attention check was placed before a randomly selected regular trial; repeated selections were permitted, allowing consecutive checks. Under all-trial counting, four distinct positions were selected from those that do not end a rating interval. Eligible locations were sampled uniformly under each rule. These placement distributions are simulation assumptions. Interval lengths were randomly ordered for each participant.

Attention-check outcomes are not specified at this stage. The schedule records their positions only. Comparisons between counting rules will assess schedule uncertainty separately from the effects of omitting attention-check rewards from the fitted predictors.


In [11]:
def make_schedule(rng, n_intervals=36, counting_rule="regular"):
    """Construct one participant's trial positions and rating boundaries."""
    assert counting_rule in ["regular", "all"]
    assert n_intervals in [35, 36]

    # start with two counted trials per interval; the remainder need three
    n_counted = 90 if counting_rule == "regular" else 94
    n_three = n_counted - 2 * n_intervals
    n_two = n_intervals - n_three
    lengths = np.array([2] * n_two + [3] * n_three)
    lengths = rng.permutation(lengths) # shuffle the order of interval lengths
    boundaries = np.cumsum(lengths) # running totals

    if counting_rule == "regular":
        # insert each check before a regular trial, never after an interval's end
        # repeated positions allow more than one check in the same gap
        before_trial = rng.integers(1, 91, size=4)
        rows = []
        for regular_order in range(1, 91):
            for _ in range(np.sum(before_trial == regular_order)):
                rows.append({
                    "is_attention_check": True,
                    "regular_trial_order": None,
                    "rating_after": False,
                })
            rows.append({
                "is_attention_check": False,
                "regular_trial_order": regular_order,
                "rating_after": regular_order in boundaries,
            })
    else:
        # choose four positions that are not immediately followed by a rating
        eligible = np.setdiff1d(np.arange(1, 95), boundaries)
        check_positions = rng.choice(eligible, size=4, replace=False)
        rows = []
        regular_order = 0
        for trial_number in range(1, 95):
            is_check = trial_number in check_positions
            if not is_check:
                regular_order += 1
            rows.append({
                "is_attention_check": is_check,
                "regular_trial_order": None if is_check else regular_order,
                "rating_after": trial_number in boundaries,
            })

    schedule = pd.DataFrame(rows)
    schedule["regular_trial_order"] = schedule["regular_trial_order"].astype("Int64")
    schedule.insert(0, "trial_number", range(1, 95))

    # a rating ends the current interval; the following trial starts the next
    schedule["interval"] = (
        schedule["rating_after"].astype(int).cumsum()
        - schedule["rating_after"].astype(int) + 1
    )
    return schedule


In [12]:
n_intervals = 36
counting_rule = "regular"
schedule_tables = []

for participant_id, offers in participant_trials.groupby("participant_id"):
    schedule = make_schedule(task_rng, n_intervals, counting_rule)
    # regular order links the schedule to the already shuffled offers
    schedule = schedule.merge(
        offers.drop(columns="participant_id"),
        on="regular_trial_order", how="left", validate="many_to_one",
    )
    schedule.insert(0, "participant_id", participant_id)
    schedule_tables.append(schedule)

scheduled_trials = pd.concat(schedule_tables, ignore_index=True)

# checks have no assigned offers yet; their missing values are intentional
scheduled_trials.head(12)


,participant_id,trial_number,is_attention_check,regular_trial_order,rating_after,interval,offer_id,trial_type,certain_value,win_value,lose_value,win_probability
0,1,1,False,1,False,1,6.0,loss,-35.0,0.0,-70.0,0.5
1,1,2,True,<NA>,False,1,NaN,NaN,NaN,NaN,NaN,NaN
2,1,3,False,2,True,1,41.0,gain,55.0,92.4,0.0,0.5
3,1,4,False,3,False,2,37.0,gain,45.0,189.0,0.0,0.5
4,1,5,False,4,True,2,3.0,gain,35.0,63.7,0.0,0.5
5,1,6,False,5,False,3,38.0,loss,-45.0,0.0,-189.0,0.5
6,1,7,False,6,False,3,10.0,loss,-35.0,0.0,-86.8,0.5
7,1,8,False,7,True,3,40.0,loss,-45.0,0.0,-225.0,0.5
8,1,9,False,8,False,4,49.0,gain,55.0,136.4,0.0,0.5
9,1,10,False,9,True,4,57.0,gain,55.0,231.0,0.0,0.5


In [13]:
# verify both counting rules and both interval counts on synthetic schedules
for rule in ["regular", "all"]:
    for count in [35, 36]:
        schedule = make_schedule(np.random.default_rng(20261122), count, rule)
        assert len(schedule) == 94
        assert schedule["is_attention_check"].sum() == 4
        assert not (schedule["is_attention_check"] & schedule["rating_after"]).any()
        assert schedule["rating_after"].sum() == count
        assert schedule["rating_after"].iloc[-1]

        regular = schedule[~schedule["is_attention_check"]]
        np.testing.assert_array_equal(regular["regular_trial_order"], np.arange(1, 91))
        counted = regular if rule == "regular" else schedule
        assert counted.groupby("interval").size().isin([2, 3]).all()

# boundaries are numbers of completed trials; zero is the initial rating
rating_boundaries = {}
for participant_id, trials in scheduled_trials.groupby("participant_id"):
    rating_boundaries[participant_id] = np.r_[
        0, trials.loc[trials["rating_after"], "trial_number"].to_numpy()
    ]

rating_boundaries


{1: array([ 0,  3,  5,  8, 10, 13, 17, 19, 21, 23, 26, 29, 32, 34, 36, 40, 43,
        46, 49, 51, 53, 55, 57, 60, 63, 66, 68, 71, 73, 76, 78, 81, 84, 86,
        88, 91, 94]),
 2: array([ 0,  3,  5,  7, 10, 12, 15, 18, 20, 24, 26, 29, 32, 35, 38, 41, 43,
        45, 47, 49, 52, 54, 57, 59, 62, 64, 66, 68, 71, 74, 77, 79, 82, 86,
        89, 91, 94])}

### 1.4 Attention-check offers and outcomes

All four attention checks used the example reported by Wang et al. (2026, p. 14): a certain gain of 20 points versus a gamble yielding 35 or 55 points with equal probability. Repeating this example across all four checks is a simulation assumption; the complete set of original check offers is unknown. Simulated participants always chose the gamble, whose two outcomes both exceed the certain reward. This represents participants who pass the checks, consistent with the study's exclusion of participants who failed a catch trial.

Gamble outcomes were drawn independently. Each check therefore has an expected value of 45 points and an RPE of −10 or +10 points, with no certain-reward contribution. Both outcomes increase cumulative points. This configuration represents four attention checks with positive outcomes.


In [14]:
# work on a copy so the schedule remains available separately
trial_data = scheduled_trials.copy()
is_check = trial_data["is_attention_check"]

# use the published example for all four checks (an explicit assumption)
trial_data.loc[is_check, "trial_type"] = "attention_check"
trial_data.loc[is_check, "certain_value"] = 20.0
trial_data.loc[is_check, "win_value"] = 55.0
trial_data.loc[is_check, "lose_value"] = 35.0
trial_data.loc[is_check, "win_probability"] = 0.5
# lose_value names the lower gamble outcome; here it is still a positive gain


In [15]:
# use the choice component stream for draws of gamble outcomes
choice_rng = np.random.default_rng(choice_seed)
n_checks = is_check.sum()
received_higher = choice_rng.random(n_checks) < 0.5

# regular-trial choices and outcomes have not yet been generated
trial_data["chose_gamble"] = pd.Series(pd.NA, index=trial_data.index, dtype="boolean")
trial_data["outcome"] = np.nan
trial_data["CR"] = np.nan
trial_data["EV"] = np.nan
trial_data["RPE"] = np.nan

# participants pass every check by choosing the dominant gamble
trial_data.loc[is_check, "chose_gamble"] = True
trial_data.loc[is_check, "outcome"] = np.where(received_higher, 55.0, 35.0)
trial_data.loc[is_check, "CR"] = 0.0
trial_data.loc[is_check, "EV"] = (
    trial_data.loc[is_check, "win_value"]
    + trial_data.loc[is_check, "lose_value"]
) / 2
trial_data.loc[is_check, "RPE"] = (
    trial_data.loc[is_check, "outcome"] - trial_data.loc[is_check, "EV"]
)

# check that the reward terms reconstruct the obtained attention-check outcomes
check_rewards = trial_data.loc[is_check, ["outcome", "CR", "EV", "RPE"]]
np.testing.assert_allclose(
    check_rewards["outcome"],
    check_rewards["CR"] + check_rewards["EV"] + check_rewards["RPE"],
)

trial_data.loc[is_check, [
    "participant_id", "trial_number", "interval", "certain_value",
    "win_value", "lose_value", "chose_gamble", "outcome", "CR", "EV", "RPE",
]]


,participant_id,trial_number,interval,certain_value,win_value,lose_value,chose_gamble,outcome,CR,EV,RPE
1,1,2,1,20.0,55.0,35.0,True,55.0,0.0,45.0,10.0
14,1,15,6,20.0,55.0,35.0,True,35.0,0.0,45.0,-10.0
38,1,39,15,20.0,55.0,35.0,True,35.0,0.0,45.0,-10.0
63,1,64,25,20.0,55.0,35.0,True,35.0,0.0,45.0,-10.0
95,2,2,1,20.0,55.0,35.0,True,55.0,0.0,45.0,10.0
114,2,21,9,20.0,55.0,35.0,True,55.0,0.0,45.0,10.0
126,2,33,13,20.0,55.0,35.0,True,55.0,0.0,45.0,10.0
176,2,83,33,20.0,55.0,35.0,True,55.0,0.0,45.0,10.0


### 1.5 Regular-trial choices, outcomes and reward terms

For the initial task-generation example, regular-trial choices were generated with a constant gambling probability of 0.5, independently of mood, offered amounts and previous outcomes. This is an illustrative simulation assumption carried forward from Notebook 6. The gambling probability was retained as a parameter. Conditional on choosing a gamble, each offered outcome was drawn with probability 0.5, as specified by the task.

Certain choices contribute the obtained certain amount to CR, with EV and RPE set to zero. Gamble choices contribute the expected gamble value to EV and the obtained outcome minus this expectation to RPE, with CR set to zero. These definitions also apply to the attention checks. The sum of CR, EV and RPE therefore reconstructs the obtained outcome on every trial.


In [16]:
gamble_probability = 0.5
assert 0 <= gamble_probability <= 1

is_regular = ~trial_data["is_attention_check"]
regular = trial_data.loc[is_regular].copy()
n_regular = len(regular)

# continue the choice stream; do not reset it after the attention-check draws
chose_gamble = choice_rng.random(n_regular) < gamble_probability
received_higher = choice_rng.random(n_regular) < regular["win_probability"].to_numpy()

# draw a potential gamble outcome independently of the choice
# it is only received if the participant chooses to gamble
potential_gamble_outcome = np.where(
    received_higher, regular["win_value"], regular["lose_value"]
)
outcome = np.where(
    chose_gamble, potential_gamble_outcome, regular["certain_value"]
)
offered_ev = (
    regular["win_probability"] * regular["win_value"]
    + (1 - regular["win_probability"]) * regular["lose_value"]
)

trial_data.loc[is_regular, "chose_gamble"] = chose_gamble
trial_data.loc[is_regular, "outcome"] = outcome
trial_data.loc[is_regular, "CR"] = np.where(
    chose_gamble, 0, regular["certain_value"]
)
trial_data.loc[is_regular, "EV"] = np.where(chose_gamble, offered_ev, 0)
trial_data.loc[is_regular, "RPE"] = np.where(chose_gamble, outcome - offered_ev, 0)

trial_data.loc[is_regular, [
    "participant_id", "trial_number", "trial_type", "certain_value",
    "win_value", "lose_value", "chose_gamble", "outcome", "CR", "EV", "RPE",
]].head(10)


,participant_id,trial_number,trial_type,certain_value,win_value,lose_value,chose_gamble,outcome,CR,EV,RPE
0,1,1,loss,-35.0,0.0,-70.0,True,0.0,0.0,-35.00,35.00
2,1,3,gain,55.0,92.4,0.0,True,92.4,0.0,46.20,46.20
3,1,4,gain,45.0,189.0,0.0,False,45.0,45.0,0.00,0.00
4,1,5,gain,35.0,63.7,0.0,True,63.7,0.0,31.85,31.85
5,1,6,loss,-45.0,0.0,-189.0,True,-189.0,0.0,-94.50,-94.50
6,1,7,loss,-35.0,0.0,-86.8,False,-35.0,-35.0,0.00,0.00
7,1,8,loss,-45.0,0.0,-225.0,False,-45.0,-45.0,0.00,0.00
8,1,9,gain,55.0,136.4,0.0,True,136.4,0.0,68.20,68.20
9,1,10,gain,55.0,231.0,0.0,True,0.0,0.0,115.50,-115.50
10,1,11,loss,-55.0,0.0,-198.0,False,-55.0,-55.0,0.00,0.00


In [19]:
# all trials, including attention checks, now have complete reward terms
assert np.isfinite(trial_data[["outcome", "CR", "EV", "RPE"]].to_numpy()).all()
np.testing.assert_allclose(
    trial_data["outcome"], trial_data["CR"] + trial_data["EV"] + trial_data["RPE"]
)

# received outcomes must belong to the chosen option
certain_choices = trial_data[~trial_data["chose_gamble"]]
gamble_choices = trial_data[trial_data["chose_gamble"]]
np.testing.assert_allclose(certain_choices["outcome"], certain_choices["certain_value"])
assert (
    np.isclose(gamble_choices["outcome"], gamble_choices["win_value"])
    | np.isclose(gamble_choices["outcome"], gamble_choices["lose_value"])
).all()

print("Outcomes match the chosen options; CR + EV + RPE reconstructs every outcome.")


Outcomes match the chosen options; CR + EV + RPE reconstructs every outcome.


### 1.6 Reward intervals

Trial-level CR, EV and RPE were summed between consecutive happiness-rating boundaries for each participant. These complete reward histories include attention-check outcomes. Each interval retained its start and end boundaries, total trial count, attention-check count and obtained-outcome sum. The initial boundary is zero, before any trial outcomes. The original trial table remains available for inspecting the contributing events.

Happiness ratings have not yet been generated. The interval sums will supply the reward inputs for the subsequent mood simulations. Any later analysis omitting attention-check rewards will preserve the original rating boundaries.


In [18]:
interval_rows = []

# collect the trials between each pair of ratings, separately for each participant
for (participant_id, interval), trials in trial_data.groupby(
    ["participant_id", "interval"]
):
    trials = trials.sort_values("trial_number")

    interval_rows.append({
        "participant_id": participant_id,
        "interval": interval,
        # boundaries count completed events, including attention checks
        "start_boundary": int(trials["trial_number"].iloc[0]) - 1,
        "end_boundary": int(trials["trial_number"].iloc[-1]),
        "n_trials": len(trials),
        "n_attention_checks": int(trials["is_attention_check"].sum()),
        "CR": trials["CR"].sum(),
        "EV": trials["EV"].sum(),
        "RPE": trials["RPE"].sum(),
        "outcome": trials["outcome"].sum(),
    })

reward_intervals = pd.DataFrame(interval_rows)
reward_intervals.head()


,participant_id,interval,start_boundary,end_boundary,n_trials,n_attention_checks,CR,EV,RPE,outcome
0,1,1,0,3,3,1,0.0,56.20,91.20,147.4
1,1,2,3,5,2,0,45.0,31.85,31.85,108.7
2,1,3,5,8,3,0,-80.0,-94.50,-94.50,-269.0
3,1,4,8,10,2,0,0.0,183.70,-47.30,136.4
4,1,5,10,13,3,0,-10.0,-87.50,-87.50,-185.0


In [20]:
# boundaries must cover every event once, with no gaps or overlap
for participant_id, intervals in reward_intervals.groupby("participant_id"):
    trials = trial_data[trial_data["participant_id"] == participant_id]
    covered_trials = []
    for row in intervals.itertuples(index=False):
        covered_trials.extend(range(row.start_boundary + 1, row.end_boundary + 1))
        assert row.n_trials == row.end_boundary - row.start_boundary

    np.testing.assert_array_equal(covered_trials, np.arange(1, len(trials) + 1))
    np.testing.assert_array_equal(
        intervals["end_boundary"], rating_boundaries[participant_id][1:]
    )
    assert intervals["n_attention_checks"].sum() == 4

# aggregation must preserve each participant's full reward totals
reward_columns = ["CR", "EV", "RPE", "outcome"]
np.testing.assert_allclose(
    reward_intervals.groupby("participant_id")[reward_columns].sum(),
    trial_data.groupby("participant_id")[reward_columns].sum(),
    rtol=1e-10, atol=1e-10,
)
np.testing.assert_allclose(
    reward_intervals["outcome"],
    reward_intervals["CR"] + reward_intervals["EV"] + reward_intervals["RPE"],
    rtol=1e-10, atol=1e-10,
)

print("Rating intervals cover every trial once and preserve the complete reward totals.")


Rating intervals cover every trial once and preserve the complete reward totals.


## 2. Clean null and recovery

*Not yet implemented.* Does the estimator recover a known linear within-participant interaction, and how often do its confidence intervals contain the generating value under null and nonzero interactions? A planned comparison will assess how omitting attention-check reward events affects recovery while retaining their effects in the generated happiness sequence.


## 3. Bounded-scale compression

*Not yet implemented.* Can non-linear compression of the happiness scale produce an apparent mood × RPE interaction when none is generated in latent mood?


## 4. Genuine moderation and RPE–EV specificity

*Not yet implemented.* Can moderation specific to RPE be distinguished from moderation of the whole obtained outcome? The generating processes and coefficient comparisons remain to be specified.


## 5. Heterogeneous RPE slopes and mood-dependent gambling (conditional)

*Conditional extension; not yet implemented.* Could differences in participants’ RPE slopes, combined with mood-dependent gambling choices, produce a misleading within-participant interaction? Inclusion of this extension will depend on progress with the preceding sections.


## 6. Summary and recommendations

*Draft.* Which findings support the proposed analysis, which assumptions do they depend on, and which limitations remain unresolved?


## 7. References

Cameron, A. C., & Miller, D. L. (2015). A practitioner’s guide to cluster-robust inference. Journal of Human Resources, 50(2), 317–372. https://doi.org/10.3368/jhr.50.2.317

Morris, T. P., White, I. R., & Crowther, M. J. (2019). Using simulation studies to evaluate statistical methods. Statistics in Medicine, 38(11), 2074–2102. https://doi.org/10.1002/sim.8086

Wang, Z., Wang, T., Xu, J., Aleman, A., Luo, Y., Blain, B., Liu, Y., & Xu, P. (2026). Dissociable Roles of Reward Prediction Error in the Contrasting Mood Dynamics of Depression and Anxiety. https://doi.org/10.7554/eLife.110631.2
